# Mini_Project: Exploratory Data Analysis (EDA) on Shark Tank India Dataset

## Submitted by_SHIMANTA NATH

#### 1.	Data Cleaning:
#### o	Check for missing values and handle them appropriately.
#### o	Identify and handle any duplicate records if present.
#### o	Convert data types if necessary.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv('Shark Tank India EDA.csv')

# Initial inspection
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nMissing Values Summary:")
print(df.isnull().sum())

In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

# 1. Deal-related columns (for pitches that didn't get deals)
deal_columns = ['deal_amount', 'deal_equity', 'deal_valuation', 
                'amount_per_shark', 'equity_per_shark']

for col in deal_columns:
    if col in df_clean.columns:
        # Fill with 0 for no-deal pitches
        df_clean[col] = df_clean[col].fillna(0)

# 2. Shark participation columns
shark_columns = [col for col in df_clean.columns if 'shark' in col.lower() and 'deal' in col.lower()]
for col in shark_columns:
    if col in df_clean.columns:
        # Fill with False for no investment
        df_clean[col] = df_clean[col].fillna(False)

# 3. Numerical columns with meaningful missing values
numerical_columns = ['pitcher_ask_amount', 'ask_equity', 'ask_valuation']
for col in numerical_columns:
    if col in df_clean.columns:
        # Use median for numerical columns
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# 4. Categorical columns
categorical_columns = ['brand_name', 'idea', 'industry']  # Add other categorical columns as needed
for col in categorical_columns:
    if col in df_clean.columns:
        # Fill with 'Unknown' for missing categorical values
        df_clean[col] = df_clean[col].fillna('Unknown')

# 5. Identifier columns - Using proper ffill() method
identifier_columns = ['episode_number', 'pitch_number']
for col in identifier_columns:
    if col in df_clean.columns:
        # Forward fill for sequential data using proper ffill() method
        df_clean[col] = df_clean[col].ffill()

# Additional cleaning: Handle any remaining missing values
# For any other numerical columns, use ffill/bfill based on context
other_numerical = df_clean.select_dtypes(include=[np.number]).columns
for col in other_numerical:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].ffill()  # Forward fill
        df_clean[col] = df_clean[col].bfill()  # Backward fill for any remaining

# For any other categorical columns
other_categorical = df_clean.select_dtypes(include=['object']).columns
for col in other_categorical:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].ffill()  # Forward fill
        df_clean[col] = df_clean[col].bfill()  # Backward fill
        df_clean[col] = df_clean[col].fillna('Unknown')  # Final fallback

In [ ]:
# For complex missing value patterns, use more sophisticated methods
from sklearn.impute import KNNImputer

# Select numerical columns for KNN imputation
numerical_cols_for_imputation = ['pitcher_ask_amount', 'ask_equity', 'ask_valuation', 
                                'deal_amount', 'deal_equity']

# Apply KNN imputer only if there are still missing values
if df_clean[numerical_cols_for_imputation].isnull().sum().sum() > 0:
    imputer = KNNImputer(n_neighbors=5)
    df_clean[numerical_cols_for_imputation] = imputer.fit_transform(df_clean[numerical_cols_for_imputation])

In [ ]:
# Verify missing values handling
print("Missing values after cleaning:")
print(df_clean.isnull().sum())

# Check data types
print("\nData types after cleaning:")
print(df_clean.dtypes)

In [ ]:
# Check for duplicate records
duplicates = df_clean.duplicated().sum()
print(f"Number of duplicate records: {duplicates}")

# Remove duplicates if any
if duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"Removed {duplicates} duplicate records")

In [ ]:
# Convert boolean columns
bool_columns = [col for col in df_clean.columns if df_clean[col].dtype == 'object' 
                and df_clean[col].nunique() == 2]

for col in bool_columns:
    df_clean[col] = df_clean[col].map({'Yes': True, 'No': False, '1': True, '0': False})

# Convert numerical columns to appropriate types
numerical_columns = ['pitcher_ask_amount', 'deal_amount', 'ask_valuation', 'deal_valuation']
for col in numerical_columns:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

In [ ]:
# Verify the cleaning
print("Missing values after cleaning:")
print(df_clean.isnull().sum())

print(f"\nTotal missing values: {df_clean.isnull().sum().sum()}")
print(f"Dataset shape: {df_clean.shape}")

# Check data types
print("\nData types:")
print(df_clean.dtypes)

In [ ]:
# Save the cleaned dataset
df_clean.to_csv('shark_tank_india_cleaned2.csv', index=False)
print("Cleaned dataset saved successfully!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")

# Load the dataset
df = pd.read_csv('shark_tank_india_cleaned2.csv')

# Display basic information about the dataset
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nMissing Values Summary:")
print(df.isnull().sum())

#### Q1.	How many deals were successfully made and what percentage of total pitches did they constitute?

In [ ]:
# Count successful deals (deal = 1)
successful_deals = df_clean[df_clean['deal'] == 1].shape[0]
total_pitches = df_clean.shape[0]
success_percentage = (successful_deals / total_pitches) * 100

print("=" * 60)
print("QUESTION 1: DEAL SUCCESS RATE ANALYSIS")
print("=" * 60)
print(f"Total number of pitches: {total_pitches}")
print(f"Number of successful deals: {successful_deals}")
print(f"Success rate: {success_percentage:.2f}%")

# Visualization
plt.figure(figsize=(10, 6))

# Pie chart
plt.subplot(1, 2, 1)
deal_counts = df_clean['deal'].value_counts()
colors = ['#ff9999', '#66b3ff']
plt.pie(deal_counts.values, labels=['No Deal', 'Deal Made'], autopct='%1.1f%%', 
        colors=colors, startangle=90)
plt.title('Deal Success Distribution')

# Bar chart
plt.subplot(1, 2, 2)
deal_data = ['No Deal', 'Deal Made']
counts = [deal_counts[0], deal_counts[1]]
bars = plt.bar(deal_data, counts, color=colors)
plt.title('Number of Pitches by Deal Outcome')
plt.ylabel('Number of Pitches')

# Add value labels on bars
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{count}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

#### Q2.	What is the average and median deal_amount for pitches that received investment?

In [ ]:
# Filter only successful deals
successful_deals_df = df_clean[df_clean['deal'] == 1]

# Calculate average and median deal amounts
average_deal_amount = successful_deals_df['deal_amount'].mean()
median_deal_amount = successful_deals_df['deal_amount'].median()

print("=" * 70)
print("QUESTION 2: DEAL AMOUNT ANALYSIS FOR SUCCESSFUL INVESTMENTS")
print("=" * 70)
print(f"Number of successful deals analyzed: {len(successful_deals_df)}")
print(f"Average deal amount: ₹{average_deal_amount:,.2f} Lakhs")
print(f"Median deal amount: ₹{median_deal_amount:,.2f} Lakhs")

# Additional statistics
min_deal_amount = successful_deals_df['deal_amount'].min()
max_deal_amount = successful_deals_df['deal_amount'].max()
std_deal_amount = successful_deals_df['deal_amount'].std()

print(f"\nAdditional Statistics:")
print(f"Minimum deal amount: ₹{min_deal_amount:,.2f} Lakhs")
print(f"Maximum deal amount: ₹{max_deal_amount:,.2f} Lakhs")
print(f"Standard deviation: ₹{std_deal_amount:,.2f} Lakhs")

#### Q3.	Which shark has made the highest number of investments? Provide a visualization of top 3 sharks.

In [ ]:
# Count investments by each shark
shark_columns = ['ashneer_deal', 'anupam_deal', 'aman_deal', 'namita_deal', 
                'vineeta_deal', 'peyush_deal', 'ghazal_deal']
shark_names = ['Ashneer', 'Anupam', 'Aman', 'Namita', 'Vineeta', 'Peyush', 'Ghazal']

shark_investments = {}
for shark_col, shark_name in zip(shark_columns, shark_names):
    shark_investments[shark_name] = df_clean[shark_col].sum()

# Get top 3 sharks
top_3_sharks = sorted(shark_investments.items(), key=lambda x: x[1], reverse=True)[:3]

print("=" * 50)
print("QUESTION 3: TOP 3 SHARKS BY NUMBER OF INVESTMENTS")
print("=" * 50)
print("Top 3 Sharks:")
for rank, (shark, count) in enumerate(top_3_sharks, 1):
    print(f"{rank}. {shark}: {count} investments")

# Visualization
plt.figure(figsize=(10, 6))
sharks, investments = zip(*top_3_sharks)
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

plt.bar(sharks, investments, color=colors, alpha=0.8)
plt.ylabel('Number of Investments')
plt.title('Top 3 Sharks by Number of Investments\n(Shark Tank India Season 1)')

# Add value labels on bars
for i, v in enumerate(investments):
    plt.text(i, v + 0.5, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

#### Q4.	What is the total amount invested by all sharks combined in entire season.

In [ ]:
# Calculate total amount invested (only for successful deals)
total_invested = successful_deals_df['deal_amount'].sum()

print("=" * 50)
print("QUESTION 4: TOTAL AMOUNT INVESTED")
print("=" * 50)
print(f"Total amount invested by all sharks: ₹{total_invested:,.0f} Lakhs")
print(f"Which is approximately: ₹{total_invested/100:,.1f} Crores")

#### Q5.	How does the deal_equity distribution look? Are there any outliers?

In [ ]:
# Filter only successful deals with positive equity
equity_data = successful_deals_df[successful_deals_df['deal_equity'] > 0]['deal_equity']

# Calculate statistics
mean_equity = equity_data.mean()
median_equity = equity_data.median()
Q1 = equity_data.quantile(0.25)
Q3 = equity_data.quantile(0.75)
IQR = Q3 - Q1
outlier_threshold = Q3 + 1.5 * IQR
outliers = equity_data[equity_data > outlier_threshold]

print("=" * 60)
print("QUESTION 5: DEAL EQUITY DISTRIBUTION & OUTLIERS")
print("=" * 60)
print(f"Equity Statistics:")
print(f"Mean: {mean_equity:.1f}% | Median: {median_equity:.1f}%")
print(f"Range: {equity_data.min():.1f}% - {equity_data.max():.1f}%")
print(f"Outliers (> {outlier_threshold:.1f}%): {len(outliers)} deals")

# Visualization
plt.figure(figsize=(12, 4))

# Histogram
plt.subplot(1, 2, 2)
plt.hist(equity_data, bins=15, color='lightgreen', edgecolor='black', alpha=0.7)
plt.axvline(mean_equity, color='red', linestyle='--', label=f'Mean: {mean_equity:.1f}%')
plt.axvline(median_equity, color='blue', linestyle='--', label=f'Median: {median_equity:.1f}%')
plt.xlabel('Equity (%)')
plt.ylabel('Frequency')
plt.title('Distribution of Deal Equity')
plt.legend()

plt.tight_layout()
plt.show()

# Show outlier deals
if len(outliers) > 0:
    print(f"\nOutlier Deals (Equity > {outlier_threshold:.1f}%):")
    outlier_deals = successful_deals_df[successful_deals_df['deal_equity'] > outlier_threshold][['brand_name', 'deal_equity', 'deal_amount']]
    for idx, row in outlier_deals.iterrows():
        print(f"  {row['brand_name']}: {row['deal_equity']}% equity for ₹{row['deal_amount']}L")

#### Q6.	Find the correlation between deal_valuation and ask_valuation. What insight can be drawn from this?

In [ ]:
# Filter only successful deals with positive valuations
valuation_data = successful_deals_df[(successful_deals_df['ask_valuation'] > 0) & 
                                   (successful_deals_df['deal_valuation'] > 0)]

# Calculate correlation
correlation = valuation_data['ask_valuation'].corr(valuation_data['deal_valuation'])

print("=" * 70)
print("QUESTION 6: CORRELATION BETWEEN DEAL VALUATION AND ASK VALUATION")
print("=" * 70)
print(f"Correlation coefficient: {correlation:.3f}")
print(f"Number of deals analyzed: {len(valuation_data)}")

# Calculate valuation changes
valuation_data = valuation_data.copy()
valuation_data['valuation_change'] = valuation_data['deal_valuation'] - valuation_data['ask_valuation']
valuation_data['valuation_change_pct'] = ((valuation_data['deal_valuation'] - valuation_data['ask_valuation']) / valuation_data['ask_valuation']) * 100

# Statistics on valuation changes
increased = len(valuation_data[valuation_data['valuation_change'] > 0])
decreased = len(valuation_data[valuation_data['valuation_change'] < 0])
unchanged = len(valuation_data[valuation_data['valuation_change'] == 0])

print(f"\nValuation Changes:")
print(f"Deals with increased valuation: {increased} ({increased/len(valuation_data)*100:.1f}%)")
print(f"Deals with decreased valuation: {decreased} ({decreased/len(valuation_data)*100:.1f}%)")
print(f"Deals with unchanged valuation: {unchanged} ({unchanged/len(valuation_data)*100:.1f}%)")

# Visualization
plt.figure(figsize=(15, 5))

# Scatter plot
plt.subplot(1, 3, 1)
plt.scatter(valuation_data['ask_valuation'], valuation_data['deal_valuation'], alpha=0.6)
plt.plot([valuation_data['ask_valuation'].min(), valuation_data['ask_valuation'].max()],
         [valuation_data['ask_valuation'].min(), valuation_data['ask_valuation'].max()], 
         'r--', alpha=0.8, label='Perfect Correlation')
plt.xlabel('Ask Valuation (₹ Lakhs)')
plt.ylabel('Deal Valuation (₹ Lakhs)')
plt.title(f'Ask vs Deal Valuation\nCorrelation: {correlation:.3f}')
plt.legend()
plt.grid(True, alpha=0.3)

# Distribution of valuation changes
plt.subplot(1, 3, 2)
plt.hist(valuation_data['valuation_change_pct'], bins=20, color='orange', alpha=0.7, edgecolor='black')
plt.axvline(0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Valuation Change (%)')
plt.ylabel('Frequency')
plt.title('Distribution of Valuation Changes')
plt.grid(True, alpha=0.3)

# Top valuation changes
print(f"\nLargest Valuation Increases:")
top_increases = valuation_data.nlargest(3, 'valuation_change_pct')[['brand_name', 'ask_valuation', 'deal_valuation', 'valuation_change_pct']]
for idx, row in top_increases.iterrows():
    print(f"  {row['brand_name']}: {row['valuation_change_pct']:+.1f}% (₹{row['ask_valuation']:.0f}L → ₹{row['deal_valuation']:.0f}L)")

print(f"\nLargest Valuation Decreases:")
top_decreases = valuation_data.nsmallest(3, 'valuation_change_pct')[['brand_name', 'ask_valuation', 'deal_valuation', 'valuation_change_pct']]
for idx, row in top_decreases.iterrows():
    print(f"  {row['brand_name']}: {row['valuation_change_pct']:+.1f}% (₹{row['ask_valuation']:.0f}L → ₹{row['deal_valuation']:.0f}L)")

#### Q7.	What is the average equity percentage given to the sharks per deal?

In [ ]:
# Calculate average equity percentage for successful deals
average_equity = successful_deals_df['deal_equity'].mean()

print("=" * 50)
print("QUESTION 7: AVERAGE EQUITY PER DEAL")
print("=" * 50)
print(f"Average equity percentage given to sharks: {average_equity:.1f}%")

# Additional statistics
median_equity = successful_deals_df['deal_equity'].median()
print(f"Median equity percentage: {median_equity:.1f}%")

# Most common equity ranges
equity_ranges = {
    'Low (1-10%)': len(successful_deals_df[(successful_deals_df['deal_equity'] >= 1) & (successful_deals_df['deal_equity'] <= 10)]),
    'Medium (11-20%)': len(successful_deals_df[(successful_deals_df['deal_equity'] > 10) & (successful_deals_df['deal_equity'] <= 20)]),
    'High (21%+)': len(successful_deals_df[successful_deals_df['deal_equity'] > 20])
}

print(f"\nEquity Distribution:")
for range_name, count in equity_ranges.items():
    percentage = (count / len(successful_deals_df)) * 100
    print(f"  {range_name}: {count} deals ({percentage:.1f}%)")

#### Q8.	Identify which episode had the highest number of deals and visualize it.

In [ ]:
# Count deals per episode
deals_per_episode = df_clean[df_clean['deal'] == 1].groupby('episode_number').size()

# Find episode with highest deals
max_episode = deals_per_episode.idxmax()
max_deals = deals_per_episode.max()

print("=" * 60)
print("QUESTION 8: EPISODE WITH HIGHEST NUMBER OF DEALS")
print("=" * 60)
print(f"Episode {max_episode} had the highest number of deals: {max_deals} deals")

# Visualization
plt.figure(figsize=(12, 6))

# Pie chart showing top episode share
plt.subplot(1, 2, 2)
total_deals = len(successful_deals_df)
other_deals = total_deals - max_deals
sizes = [max_deals, other_deals]
labels = [f'Episode {max_episode}\n({max_deals} deals)', f'Other Episodes\n({other_deals} deals)']
colors = ['red', 'lightblue']
plt.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
plt.title(f'Episode {max_episode} Share of Total Deals')

plt.tight_layout()
plt.show()

# Show top 3 episodes
print(f"\nTop 3 Episodes by Number of Deals:")
top_3_episodes = deals_per_episode.nlargest(3)
for ep, deals in top_3_episodes.items():
    print(f"  Episode {ep}: {deals} deals")

#### Q9.	How many pitches asked for more than ₹1 crore and how many of those received investments?

In [ ]:
# Convert ask amounts - note: 1 crore = 100 lakhs
pitches_over_1cr = df_clean[df_clean['pitcher_ask_amount'] > 100]
successful_over_1cr = pitches_over_1cr[pitches_over_1cr['deal'] == 1]

total_over_1cr = len(pitches_over_1cr)
successful_over_1cr_count = len(successful_over_1cr)
success_rate = (successful_over_1cr_count / total_over_1cr) * 100 if total_over_1cr > 0 else 0

print("=" * 70)
print("QUESTION 9: PITCHES ASKING FOR MORE THAN ₹1 CRORE")
print("=" * 70)
print(f"Pitches asking for > ₹1 crore: {total_over_1cr}")
print(f"Successful deals among them: {successful_over_1cr_count}")
print(f"Success rate for > ₹1 crore asks: {success_rate:.1f}%")

# Compare with overall success rate
overall_success_rate = (len(successful_deals_df) / len(df_clean)) * 100
print(f"Overall success rate: {overall_success_rate:.1f}%")

# Show the successful big asks
if successful_over_1cr_count > 0:
    print(f"\nSuccessful > ₹1 crore deals:")
    for idx, row in successful_over_1cr.iterrows():
        print(f"  {row['brand_name']}: Asked ₹{row['pitcher_ask_amount']:.0f}L, Got ₹{row['deal_amount']:.0f}L")

#### Q10.	What percentage of pitches involved more than one shark investing together?

In [ ]:
# Count pitches with multiple sharks (total_sharks_invested > 1)
multi_shark_deals = successful_deals_df[successful_deals_df['total_sharks_invested'] > 1]
single_shark_deals = successful_deals_df[successful_deals_df['total_sharks_invested'] == 1]

multi_shark_count = len(multi_shark_deals)
single_shark_count = len(single_shark_deals)
total_successful_deals = len(successful_deals_df)

multi_shark_percentage = (multi_shark_count / total_successful_deals) * 100

print("=" * 70)
print("QUESTION 10: MULTIPLE SHARK INVESTMENTS")
print("=" * 70)
print(f"Total successful deals: {total_successful_deals}")
print(f"Deals with multiple sharks: {multi_shark_count}")
print(f"Deals with single shark: {single_shark_count}")
print(f"Percentage with multiple sharks: {multi_shark_percentage:.1f}%")

# Breakdown by number of sharks
shark_count_distribution = successful_deals_df['total_sharks_invested'].value_counts().sort_index()
print(f"\nDistribution by number of sharks:")
for sharks, count in shark_count_distribution.items():
    percentage = (count / total_successful_deals) * 100
    print(f"  {sharks} shark(s): {count} deals ({percentage:.1f}%)")

#### Q11.	How does the investment behavior of Ashneer Grover compare with Peyush Bansal in terms of total amount invested?

In [ ]:
# Calculate total amount invested by each shark
ashneer_deals = successful_deals_df[successful_deals_df['ashneer_deal'] == 1]
peyush_deals = successful_deals_df[successful_deals_df['peyush_deal'] == 1]

ashneer_total = ashneer_deals['amount_per_shark'].sum() * ashneer_deals['ashneer_deal'].sum()
peyush_total = peyush_deals['amount_per_shark'].sum() * peyush_deals['peyush_deal'].sum()

ashneer_avg = ashneer_deals['amount_per_shark'].mean() if len(ashneer_deals) > 0 else 0
peyush_avg = peyush_deals['amount_per_shark'].mean() if len(peyush_deals) > 0 else 0

print("=" * 70)
print("QUESTION 11: ASHNEER vs PEYUSH - INVESTMENT COMPARISON")
print("=" * 70)
print(f"Ashneer Grover:")
print(f"  Total invested: ₹{ashneer_total:,.0f}L")
print(f"  Number of deals: {len(ashneer_deals)}")
print(f"  Average per deal: ₹{ashneer_avg:,.0f}L")

print(f"\nPeyush Bansal:")
print(f"  Total invested: ₹{peyush_total:,.0f}L")
print(f"  Number of deals: {len(peyush_deals)}")
print(f"  Average per deal: ₹{peyush_avg:,.0f}L")

print(f"\nComparison:")
print(f"  Difference: ₹{peyush_total - ashneer_total:,.0f}L")
print(f"  Peyush invested {peyush_total/ashneer_total:.1f}x more than Ashneer")

#### Q12.	Create a box plot to analyze amount_per_shark. What insights can be gathered?

In [ ]:
# Filter only deals where sharks invested (amount_per_shark > 0)
shark_investment_data = successful_deals_df[successful_deals_df['amount_per_shark'] > 0]['amount_per_shark']

# Calculate statistics
stats = shark_investment_data.describe()
Q1 = stats['25%']
Q3 = stats['75%']
IQR = Q3 - Q1

print("=" * 50)
print("QUESTION 12: AMOUNT PER SHARK BOX PLOT")
print("=" * 50)

# Box plot
plt.figure(figsize=(8, 6))
plt.boxplot(shark_investment_data, vert=True, patch_artist=True)
plt.ylabel('Amount per Shark (₹ Lakhs)')
plt.title('Investment Amount per Shark\n(Box Plot Analysis)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Key Insights:")
print(f"• Typical investment: ₹{Q1:.0f}L-₹{Q3:.0f}L per shark")
print(f"• Median: ₹{stats['50%']:.0f}L per shark")
print(f"• Right-skewed: Few large investments pull mean upward")
print(f"• Conservative approach: Most deals under ₹25L per shark")

#### Q13.	Are there any cases where the deal_amount exceeded the pitcher_ask_amount? If yes, list those cases.

In [ ]:
## Find cases where deal_amount > pitcher_ask_amount
better_deals = successful_deals_df[successful_deals_df['deal_amount'] > successful_deals_df['pitcher_ask_amount']]

print("=" * 70)
print("QUESTION 13: DEALS WHERE FINAL AMOUNT EXCEEDED ASK AMOUNT")
print("=" * 70)

if len(better_deals) > 0:
    print(f"Number of cases: {len(better_deals)}")
    print(f"\nCases where deal amount exceeded ask amount:")
    for idx, row in better_deals.iterrows():
        increase_pct = ((row['deal_amount'] - row['pitcher_ask_amount']) / row['pitcher_ask_amount']) * 100
        print(f"  {row['brand_name']}: Asked ₹{row['pitcher_ask_amount']}L, Got ₹{row['deal_amount']}L (+{increase_pct:.0f}%)")
else:
    print("No cases found where deal amount exceeded ask amount.")

#### Q14.	Which shark has the highest return on investment (ROI) based on deal_amount vs. deal_equity?

In [ ]:
# Calculate ROI for each shark (deal_valuation = deal_amount / (deal_equity/100))
shark_roi = {}
shark_names = ['Ashneer', 'Anupam', 'Aman', 'Namita', 'Vineeta', 'Peyush', 'Ghazal']
shark_columns = ['ashneer_deal', 'anupam_deal', 'aman_deal', 'namita_deal', 'vineeta_deal', 'peyush_deal', 'ghazal_deal']

for shark_col, shark_name in zip(shark_columns, shark_names):
    shark_deals = successful_deals_df[successful_deals_df[shark_col] == 1]
    if len(shark_deals) > 0:
        # Calculate average deal valuation per lakh invested
        total_invested = shark_deals['amount_per_shark'].sum()
        total_valuation = shark_deals['deal_valuation'].sum()
        if total_invested > 0:
            roi = total_valuation / total_invested
            shark_roi[shark_name] = roi

# Find shark with highest ROI
highest_roi_shark = max(shark_roi.items(), key=lambda x: x[1])

print("=" * 60)
print("QUESTION 14: SHARK WITH HIGHEST ROI")
print("=" * 60)
print(f"Highest ROI: {highest_roi_shark[0]} with {highest_roi_shark[1]:.1f}x return")

print(f"\nAll Sharks ROI (Valuation per ₹1L invested):")
for shark, roi in sorted(shark_roi.items(), key=lambda x: x[1], reverse=True):
    print(f"  {shark}: {roi:.1f}x")

#### Q15.	Identify trends in equity distribution—are sharks investing in lower or higher equity stakes over time?

In [ ]:
# Calculate average equity by episode over time
equity_trend = successful_deals_df.groupby('episode_number')['deal_equity'].mean()

# Calculate correlation between episode number and equity
correlation = successful_deals_df['episode_number'].corr(successful_deals_df['deal_equity'])

print("=" * 70)
print("QUESTION 15: EQUITY DISTRIBUTION TRENDS OVER TIME")
print("=" * 70)
print(f"Correlation between episode number and equity: {correlation:.3f}")

# Visualization
plt.figure(figsize=(12, 5))

# Trend line
plt.subplot(1, 2, 1)
plt.plot(equity_trend.index, equity_trend.values, marker='o', linewidth=2, markersize=6)
plt.xlabel('Episode Number')
plt.ylabel('Average Equity (%)')
plt.title('Equity Trend Over Episodes')
plt.grid(True, alpha=0.3)

# Scatter plot with trend line
plt.subplot(1, 2, 2)
plt.scatter(successful_deals_df['episode_number'], successful_deals_df['deal_equity'], alpha=0.6)
z = np.polyfit(successful_deals_df['episode_number'], successful_deals_df['deal_equity'], 1)
p = np.poly1d(z)
plt.plot(successful_deals_df['episode_number'], p(successful_deals_df['episode_number']), "r--", alpha=0.8)
plt.xlabel('Episode Number')
plt.ylabel('Deal Equity (%)')
plt.title('Equity Distribution Over Time\nwith Trend Line')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare early vs late episodes
early_episodes = successful_deals_df[successful_deals_df['episode_number'] <= 17]
late_episodes = successful_deals_df[successful_deals_df['episode_number'] > 17]

early_avg_equity = early_episodes['deal_equity'].mean()
late_avg_equity = late_episodes['deal_equity'].mean()

print(f"\nEarly Episodes (1-17): {early_avg_equity:.1f}% average equity")
print(f"Late Episodes (18-35): {late_avg_equity:.1f}% average equity")
print(f"Change: {late_avg_equity - early_avg_equity:+.1f}%")

#### Q16.	What is the relationship between pitcher_ask_amount and deal_amount? Do pitchers who ask for less tend to secure more deals?

In [ ]:
# Calculate correlation between ask amount and deal amount for successful deals
correlation_successful = successful_deals_df['pitcher_ask_amount'].corr(successful_deals_df['deal_amount'])

# Analyze success rates by ask amount ranges
ask_ranges = [0, 25, 50, 75, 1000]  # 0-25L, 26-50L, 51-75L, 76L+
range_labels = ['0-25L', '26-50L', '51-75L', '76L+']

success_rates = []
for i in range(len(ask_ranges)-1):
    low = ask_ranges[i]
    high = ask_ranges[i+1]
    range_pitches = df_clean[(df_clean['pitcher_ask_amount'] > low) & (df_clean['pitcher_ask_amount'] <= high)]
    range_success = range_pitches[range_pitches['deal'] == 1]
    success_rate = (len(range_success) / len(range_pitches)) * 100 if len(range_pitches) > 0 else 0
    success_rates.append(success_rate)

print("=" * 70)
print("QUESTION 16: ASK AMOUNT vs DEAL AMOUNT RELATIONSHIP")
print("=" * 70)
print(f"Correlation (successful deals): {correlation_successful:.3f}")

print(f"\nSuccess Rates by Ask Amount Range:")
for label, rate in zip(range_labels, success_rates):
    print(f"  {label}: {rate:.1f}% success rate")

# Visualization
plt.figure(figsize=(15, 5))

# Success rates by ask amount range
plt.subplot(1, 3, 2)
colors = ['green', 'lightgreen', 'orange', 'red']
bars = plt.bar(range_labels, success_rates, color=colors, alpha=0.7)
plt.xlabel('Ask Amount Range')
plt.ylabel('Success Rate (%)')
plt.title('Success Rate by Ask Amount Range')
for bar, rate in zip(bars, success_rates):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{rate:.1f}%', 
             ha='center', va='bottom')

# Statistical comparison
avg_ask_successful = successful_asks.mean()
avg_ask_unsuccessful = unsuccessful_asks.mean()

print(f"\nAverage Ask Amount:")
print(f"  Successful pitches: ₹{avg_ask_successful:.1f}L")
print(f"  Unsuccessful pitches: ₹{avg_ask_unsuccessful:.1f}L")

#### Q17.	Identify if there is a pattern in episode-wise deal closures—do certain episodes see more investments?

In [ ]:
# Count deals and pitches per episode
deals_per_episode = df_clean[df_clean['deal'] == 1].groupby('episode_number').size()
pitches_per_episode = df_clean.groupby('episode_number').size()
success_rate_per_episode = (deals_per_episode / pitches_per_episode * 100).fillna(0)

print("=" * 70)
print("QUESTION 17: EPISODE-WISE DEAL CLOSURE PATTERNS")
print("=" * 70)

# Find episodes with highest and lowest deal counts
max_episode = deals_per_episode.idxmax()
min_episode = deals_per_episode.idxmin()
max_deals = deals_per_episode.max()
min_deals = deals_per_episode.min()

print(f"Episode with most deals: Episode {max_episode} ({max_deals} deals)")
print(f"Episode with fewest deals: Episode {min_episode} ({min_deals} deals)")

# Visualization
plt.figure(figsize=(15, 5))

# Deals per episode
plt.subplot(1, 3, 1)
episodes = deals_per_episode.index
colors = ['red' if x == max_deals else 'lightblue' for x in deals_per_episode.values]
plt.bar(episodes, deals_per_episode.values, color=colors, alpha=0.8)
plt.xlabel('Episode Number')
plt.ylabel('Number of Deals')
plt.title('Deals per Episode')
plt.xticks(episodes)

# Success rate per episode
plt.subplot(1, 3, 2)
plt.bar(success_rate_per_episode.index, success_rate_per_episode.values, color='green', alpha=0.7)
plt.xlabel('Episode Number')
plt.ylabel('Success Rate (%)')
plt.title('Success Rate per Episode')
plt.xticks(episodes)

# Cumulative deals over time
plt.subplot(1, 3, 3)
cumulative_deals = deals_per_episode.cumsum()
plt.plot(cumulative_deals.index, cumulative_deals.values, marker='o', linewidth=2)
plt.xlabel('Episode Number')
plt.ylabel('Cumulative Deals')
plt.title('Cumulative Deals Over Season')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical analysis
print(f"\nEpisode Deal Statistics:")
print(f"Average deals per episode: {deals_per_episode.mean():.1f}")
print(f"Standard deviation: {deals_per_episode.std():.1f}")
print(f"Most common: {deals_per_episode.mode().values[0]} deals per episode")

# Identify patterns
high_deal_episodes = deals_per_episode[deals_per_episode > deals_per_episode.mean()]
low_deal_episodes = deals_per_episode[deals_per_episode < deals_per_episode.mean()]

print(f"\nPattern Analysis:")
print(f"High-deal episodes (> average): {len(high_deal_episodes)} episodes")
print(f"Low-deal episodes (< average): {len(low_deal_episodes)} episodes")

#### Q18.	Analyze the impact of equity dilution on deal_valuation. Are lower equity asks leading to higher deal valuations?

In [ ]:
# Calculate correlation between deal_equity and deal_valuation
correlation = successful_deals_df['deal_equity'].corr(successful_deals_df['deal_valuation'])

# Group by equity ranges and analyze valuations
equity_ranges = [0, 5, 10, 15, 20, 100]  # 0-5%, 6-10%, 11-15%, 16-20%, 21%+
range_labels = ['0-5%', '6-10%', '11-15%', '16-20%', '21%+']

valuation_by_equity = []
for i in range(len(equity_ranges)-1):
    low = equity_ranges[i]
    high = equity_ranges[i+1]
    range_deals = successful_deals_df[(successful_deals_df['deal_equity'] > low) & (successful_deals_df['deal_equity'] <= high)]
    avg_valuation = range_deals['deal_valuation'].mean() if len(range_deals) > 0 else 0
    valuation_by_equity.append(avg_valuation)

print("=" * 70)
print("QUESTION 18: EQUITY DILUTION vs DEAL VALUATION")
print("=" * 70)
print(f"Correlation between equity and valuation: {correlation:.3f}")

print(f"\nAverage Valuation by Equity Range:")
for label, valuation in zip(range_labels, valuation_by_equity):
    print(f"  {label} equity: ₹{valuation:,.0f}L average valuation")

# Visualization
plt.figure(figsize=(12, 5))

# Scatter plot: Equity vs Valuation
plt.subplot(1, 2, 1)
plt.scatter(successful_deals_df['deal_equity'], successful_deals_df['deal_valuation'], alpha=0.6)
plt.xlabel('Deal Equity (%)')
plt.ylabel('Deal Valuation (₹ Lakhs)')
plt.title(f'Equity vs Valuation\nCorrelation: {correlation:.3f}')
plt.grid(True, alpha=0.3)

# Bar chart: Average valuation by equity range
plt.subplot(1, 2, 2)
colors = ['green', 'lightgreen', 'yellow', 'orange', 'red']
bars = plt.bar(range_labels, valuation_by_equity, color=colors, alpha=0.7)
plt.xlabel('Equity Range')
plt.ylabel('Average Valuation (₹ Lakhs)')
plt.title('Average Valuation by Equity Range')

# Add value labels on bars
for bar, valuation in zip(bars, valuation_by_equity):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f'₹{valuation:.0f}L', 
             ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Analyze top vs bottom quartiles
top_quartile_equity = successful_deals_df['deal_equity'].quantile(0.75)
bottom_quartile_equity = successful_deals_df['deal_equity'].quantile(0.25)

high_equity_deals = successful_deals_df[successful_deals_df['deal_equity'] >= top_quartile_equity]
low_equity_deals = successful_deals_df[successful_deals_df['deal_equity'] <= bottom_quartile_equity]

avg_valuation_high_equity = high_equity_deals['deal_valuation'].mean()
avg_valuation_low_equity = low_equity_deals['deal_valuation'].mean()

print(f"\nQuartile Analysis:")
print(f"High equity deals (≥{top_quartile_equity:.1f}%): ₹{avg_valuation_high_equity:,.0f}L avg valuation")
print(f"Low equity deals (≤{bottom_quartile_equity:.1f}%): ₹{avg_valuation_low_equity:,.0f}L avg valuation")
print(f"Valuation difference: ₹{avg_valuation_low_equity - avg_valuation_high_equity:,.0f}L")